### Imports des bibliothèques et configuration 

In [22]:
import torch
torch.cuda.is_available()

True

In [23]:
import sys
!{sys.executable} -m pip install pyyaml scipy pandas scikit-learn tqdm torch


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Home\mon_env_gpu\Scripts\python.exe -m pip install --upgrade pip


In [24]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse as sp
import subprocess
import torch
from types import SimpleNamespace
from sklearn.metrics import normalized_mutual_info_score
from tqdm.auto import tqdm
from sklearn.metrics import cluster
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

In [34]:
# --- CONFIGURATION DES CHEMINS ---
REPO_DIR = r"C:\Users\Home\Documents\M2\Projet_PPD\output\ECRTM" 
DATA_ROOT = r"C:\Users\Home\Documents\M2\Projet_PPD\data"
OUT_DIR_ECRTM = r"C:\Users\Home\Documents\M2\Projet_PPD\output\ECRTM"

# Chemins vers les fichiers de configuration
CONFIGS = {
    "20NG": os.path.join(REPO_DIR, "configs", "model", "ECRTM_20NG.yaml"),
    "AGNews": os.path.join(REPO_DIR, "configs", "model", "ECRTM_AGNews.yaml"),
    "IMDB": os.path.join(REPO_DIR, "configs", "model", "ECRTM_IMDB.yaml"),
    "YahooAnswer": os.path.join(REPO_DIR, "configs", "model", "ECRTM_YahooAnswer.yaml"),
}

if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

# Paramètres globaux
DATASETS = ["20NG", "AGNews", "IMDB", "YahooAnswer"]
K_VALUES = [20, 50, 100]
SEED = 42

### Importation du modèle ECRTM et utilitaires de chargement de données

In [35]:
sys.path.append(REPO_DIR)
from models.ECRTM import ECRTM

def load_cfg(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)  

def load_vocab(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_bow_npz(path):
    X = sp.load_npz(path).astype(np.float32)  
    return X.toarray()                       

def load_word_emb_npz(path):
    W = sp.load_npz(path)                 
    return W.astype(np.float32).toarray()

### Configuration des arguments et extraction des topics

In [36]:
def build_args_from_yaml(cfg, K, vocab_size, word_emb):
    cfg = dict(cfg)  
    cfg["num_topic"] = K
    cfg["vocab_size"] = vocab_size
    cfg["word_embeddings"] = word_emb
    return SimpleNamespace(**cfg)

def infer_theta(model, bow_np, device, batch_size=256):
    model.eval()
    thetas = []
    with torch.no_grad():
        X = torch.from_numpy(bow_np).float()
        loader = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=False)
        for (bow,) in loader:
            bow = bow.to(device)
            theta = model.get_theta(bow)  
            thetas.append(theta.cpu().numpy())
    return np.vstack(thetas)


### Pipeline d'entraînement et d'exportation du modèle

In [28]:
def train_one(dataset, K, seed=42):
    set_seed(seed)

    data_dir = os.path.join(DATA_ROOT, dataset)
    cfg = load_cfg(CONFIGS[dataset])

    # Hyperparams depuis YAML
    epochs = int(cfg["epochs"])
    batch_size = int(cfg["batch_size"])
    lr = float(cfg["learning_rate"])

    # Data
    train_bow = load_bow_npz(os.path.join(data_dir, "train_bow.npz"))
    test_bow  = load_bow_npz(os.path.join(data_dir, "test_bow.npz"))
    vocab     = load_vocab(os.path.join(data_dir, "vocab.txt"))
    word_emb  = load_word_emb_npz(os.path.join(data_dir, "word_embeddings.npz"))

    V = train_bow.shape[1]
    assert V == test_bow.shape[1] == len(vocab) == word_emb.shape[0], "Mismatch vocab sizes"


    args = build_args_from_yaml(cfg, K=K, vocab_size=V, word_emb=word_emb)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = ECRTM(args).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    X = torch.from_numpy(train_bow).float()
    loader = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=True)

    epoch_bar = tqdm(range(1, epochs + 1), desc=f"{dataset} | K={K} | seed={seed}", position=0)
    for epoch in epoch_bar:
        model.train()
        total_loss = 0.0

        batch_bar = tqdm(loader, desc=f"epoch {epoch}/{epochs}", leave=False, position=1)
        for (bow,) in batch_bar:
            bow = bow.to(device)
            rst = model(bow)
            loss = rst["loss"]

            opt.zero_grad()
            loss.backward()
            opt.step()

            l = float(loss.detach().cpu())
            total_loss += l
            batch_bar.set_postfix(loss=l)

        epoch_bar.set_postfix(avg_loss=total_loss / len(loader))

    model.eval()
    with torch.no_grad():
        beta = model.get_beta().cpu().numpy()

    train_theta = infer_theta(model, train_bow, device=device, batch_size=256)
    test_theta  = infer_theta(model, test_bow,  device=device, batch_size=256)

    out_path = os.path.join(OUT_DIR, f"{dataset}_ECRTM_K{K}_seed{seed}.mat")
    scipy.io.savemat(out_path, {"beta": beta, "train_theta": train_theta, "test_theta": test_theta})
    print("Saved:", out_path)
    return out_path


### Fonctions de calcul des métriques

In [29]:
def calculate_gensim_cv(topics_list, texts):
    if not topics_list or not texts: return 0.0
    dictionary = Dictionary(texts)
    cm = CoherenceModel(topics=topics_list, texts=texts, dictionary=dictionary, coherence='c_v')
    return cm.get_coherence()

def calculate_topic_diversity(topics_list):
    if not topics_list: return 0.0
    all_words = [word for topic in topics_list for word in topic]
    return len(set(all_words)) / len(all_words)

def purity_score(y_true, y_pred):
    contingency_matrix = cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.max(contingency_matrix, axis=0)) / np.sum(contingency_matrix)

### Pipeline de traitement

In [30]:
def process_dataset_ecrtm(dataset, K):
    file_name = f"{dataset}_ECRTM_K{K}_seed42.mat"
    mat_path = os.path.join(OUT_DIR_ECRTM, dataset, file_name)
    
    if not os.path.exists(mat_path):
        return None

    data = scipy.io.loadmat(mat_path)
    needs_update = False
    cv = data.get('C_V', [[None]])[0][0]
    td = data.get('TD', [[None]])[0][0]
    purity = data.get('Purity', [[None]])[0][0]
    nmi = data.get('NMI', [[None]])[0][0]

    # Si une des métriques est manquante, lancement des calculs
    if cv is None or td is None or purity is None or nmi is None:
        print(f" -> Calcul des métriques manquantes pour {dataset} K={K}...")
        
        # 1. Préparation des données 
        vocab_path = os.path.join(DATA_ROOT, dataset, "vocab.txt")
        corpus_path = os.path.join(DATA_ROOT, dataset, "test_texts.txt")
        
        with open(vocab_path, "r", encoding="utf-8") as f:
            vocab = [line.strip() for line in f if line.strip()]
        
        # Extraction des thèmes pour CV et TD
        beta = data['beta']
        topics_list = [[vocab[i] for i in np.argsort(beta[k])[::-1][:10]] for k in range(K)]

        # 2. Calcul des métriques si elles manquent
        if cv is None:
            if os.path.exists(corpus_path):
                with open(corpus_path, "r", encoding="utf-8") as f:
                    texts = [line.strip().split() for line in f]
                cv = calculate_gensim_cv(topics_list, texts)
            else: cv = 0.0
        
        if td is None:
            td = calculate_topic_diversity(topics_list)
            
        if purity is None or nmi is None:
            labels_path = os.path.join(DATA_ROOT, dataset, "test_labels.txt")
            y_true = np.loadtxt(labels_path, dtype=int)
            _, theta = find_theta_in_mat(data, K, len(y_true))
            y_pred = theta.argmax(axis=1)
            from sklearn.metrics import normalized_mutual_info_score
            nmi = normalized_mutual_info_score(y_true, y_pred)
            purity = purity_score(y_true, y_pred)

        # --- SAUVEGARDE DANS LE FICHIER .MAT ---
        data['C_V'] = cv
        data['TD'] = td
        data['Purity'] = purity
        data['NMI'] = nmi
        scipy.io.savemat(mat_path, data)
        print(f" Métriques sauvegardées dans le .mat.")
    else:
        print(f" Skip {dataset} (K={K}) : déjà calculé.")

    return {
        "Dataset": dataset, "K": K,
        "C_V": round(float(cv), 4), "TD": round(float(td), 4),
        "Purity": round(float(purity), 4), "NMI": round(float(nmi), 4)
    }

### Fonction pour la matrice theta 

In [31]:
def find_theta_in_mat(data, K, num_docs):
    for key in ['theta', 'p_z_d', 'test_theta', 'theta_test']:
        if key in data:
            theta = data[key]
            if theta.shape == (num_docs, K) or theta.shape == (K, num_docs):
                if theta.shape == (K, num_docs):
                    theta = theta.T
                return key, theta
    
    for key, value in data.items():
        if isinstance(value, np.ndarray):
            if value.shape == (num_docs, K):
                return key, value
            if value.shape == (K, num_docs):
                return key, value.T
                
    raise ValueError(f"Impossible de trouver la matrice theta dans le fichier .mat. Clés dispos : {list(data.keys())}")

### Exécution finale 

In [32]:
final_summary = []

for dataset in DATASETS:
    print(f"\n--- Analyse : {dataset} ---")
    for K in K_VALUES:
        res = process_dataset_ecrtm(dataset, K)
        if res:
            final_summary.append(res)

if final_summary:
    df_results = pd.DataFrame(final_summary)
    csv_output = os.path.join(OUT_DIR_ECRTM, "ECRTM_ALL_METRICS.csv")
    df_results.to_csv(csv_output, index=False)
    print(f"\n--- TERMINE ---")
    print(f"Tableau sauvegardé ici : {csv_output}")
    
display(df_results.pivot(index="Dataset", columns="K", values=["C_V", "TD", "Purity", "NMI"]))


--- Analyse : 20NG ---
 Skip 20NG (K=20) : déjà calculé.
 Skip 20NG (K=50) : déjà calculé.
 Skip 20NG (K=100) : déjà calculé.

--- Analyse : AGNews ---
 Skip AGNews (K=20) : déjà calculé.
 Skip AGNews (K=50) : déjà calculé.
 Skip AGNews (K=100) : déjà calculé.

--- Analyse : IMDB ---
 Skip IMDB (K=20) : déjà calculé.
 Skip IMDB (K=50) : déjà calculé.
 Skip IMDB (K=100) : déjà calculé.

--- Analyse : YahooAnswer ---
 Skip YahooAnswer (K=20) : déjà calculé.
 Skip YahooAnswer (K=50) : déjà calculé.
 Skip YahooAnswer (K=100) : déjà calculé.

--- TERMINE ---
Tableau sauvegardé ici : C:\Users\Home\Documents\M2\Projet_PPD\output\ECRTM\ECRTM_ALL_METRICS.csv


C_V                     TD                Purity          \
K               20      50      100    20     50     100     20      50    
Dataset                                                                    
20NG         0.6742  0.4541  0.4607  0.935  0.970  0.912  0.5451  0.5625   
AGNews       0.4360  0.4927  0.5406  0.855  1.000  1.000  0.4476  0.6316   
IMDB         0.5472  0.4000  0.3825  0.985  0.974  0.905  0.6999  0.6749   
YahooAnswer  0.4518  0.4689  0.4869  0.995  1.000  0.917  0.5452  0.5288   

                        NMI                  
K               100     20      50      100  
Dataset                                      
20NG         0.5328  0.5532  0.5125  0.4881  
AGNews       0.3952  0.1776  0.2683  0.2077  
IMDB         0.7062  0.0797  0.0506  0.0524  
YahooAnswer  0.5580  0.3185  0.2912  0.3077

### Comparaison entre les résultats de l'article et de nos reproductions

In [33]:
paper_ecrtm = {
    "20NG":    {"K50":  {"CV": 0.431, "TD": 0.964, "Purity": 0.560, "NMI": 0.524},
                "K100": {"CV": 0.405, "TD": 0.904, "Purity": 0.555, "NMI": 0.494}},
    "IMDB":    {"K50":  {"CV": 0.393, "TD": 0.974, "Purity": 0.694, "NMI": 0.058},
                "K100": {"CV": 0.373, "TD": 0.887, "Purity": 0.694, "NMI": 0.049}},
    "Yahoo":   {"K50":  {"CV": 0.405, "TD": 0.985, "Purity": 0.550, "NMI": 0.295},
                "K100": {"CV": 0.389, "TD": 0.903, "Purity": 0.563, "NMI": 0.311}},
    "AGNews":  {"K50":  {"CV": 0.466, "TD": 0.961, "Purity": 0.802, "NMI": 0.367},
                "K100": {"CV": 0.416, "TD": 0.981, "Purity": 0.812, "NMI": 0.428}},
}


repro_ecrtm = {
    "20NG": {
        "K50":  {"CV": 0.454, "TD": 0.970, "Purity": 0.563, "NMI": 0.513},
        "K100": {"CV": 0.461, "TD": 0.912, "Purity": 0.533, "NMI": 0.488}
    },
    "AGNews": {
        "K50":  {"CV": 0.493, "TD": 1.000, "Purity": 0.632, "NMI": 0.268},
        "K100": {"CV": 0.541, "TD": 1.000, "Purity": 0.395, "NMI": 0.208}
    },
    "IMDB": {
        "K50":  {"CV": 0.400, "TD": 0.974, "Purity": 0.675, "NMI": 0.051},
        "K100": {"CV": 0.383, "TD": 0.905, "Purity": 0.706, "NMI": 0.052}
    },
    "Yahoo": {
        "K50":  {"CV": 0.469, "TD": 1.000, "Purity": 0.529, "NMI": 0.291},
        "K100": {"CV": 0.487, "TD": 0.917, "Purity": 0.558, "NMI": 0.308}
    }
}
data = []
metrics = ["CV", "TD", "Purity", "NMI"]

for ds in ["20NG", "IMDB", "Yahoo", "AGNews"]:
    for k_key in ["K50", "K100"]:
        row = {"Dataset": ds, "K": int(k_key[1:])}
        for m in metrics:
            p_val = paper_ecrtm[ds][k_key][m]
            r_val = repro_ecrtm[ds][k_key][m]
            row[f"{m}_Papier"] = p_val
            row[f"{m}_Repro"] = r_val
            row[f"{m}_Ecart"] = round(r_val - p_val, 3)
        data.append(row)

df = pd.DataFrame(data).set_index(["Dataset", "K"])

display(df.style
    .format("{:.3f}")
    .set_caption("ECRTM — Papier vs Repro")
)